<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [3]</a>'.</span>

# APH/PPH Split

This notebook checks the APH/PPH split and hemorrhage logic:

1. APH and PPH incidence are in the ballpark of artifact expectations
2. Only severe hemorrhage cases die from hemorrhage
3. Misoprostol affects postpartum hemorrhage incidence but not antepartum hemorrhage incidence

In [1]:
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd

import vivarium_gates_mncnh
from vivarium import Artifact, InteractiveContext
from vivarium.framework.configuration import build_model_specification

from vivarium_gates_mncnh.constants.data_keys import MATERNAL_HEMORRHAGE
from vivarium_gates_mncnh.constants.data_values import (
    ANEMIA_THRESHOLDS,
    COLUMNS,
    HEMORRHAGE_SEVERITY,
    SIMULATION_EVENT_NAMES,
    SIMULATION_STEPS,
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [2]:
# Build a small interactive simulation for quick checks
model_spec_path = Path(vivarium_gates_mncnh.__file__).parent / 'model_specifications/model_spec.yaml'
base_spec = build_model_specification(model_spec_path)

DRAW = int(base_spec.configuration.input_data.input_draw_number)
POP_SIZE = 60_000

STEP_MAPPER = {name: i + 1 for i, name in enumerate(SIMULATION_STEPS)}

def make_spec(scenario: str = 'baseline', pop_size: int = POP_SIZE):
    spec = deepcopy(base_spec)
    spec.configuration.population.population_size = pop_size
    spec.configuration.intervention.scenario = scenario
    return spec

def make_sim(scenario: str = 'baseline', pop_size: int = POP_SIZE):
    return InteractiveContext(make_spec(scenario=scenario, pop_size=pop_size))

# TODO tdy fix this to use a while loop
def run_to_step(sim: InteractiveContext, step_name: str):
    sim.take_steps(STEP_MAPPER[step_name])
    return sim

artifact = Artifact(base_spec.configuration.input_data.artifact_path)
draw_col = f'draw_{DRAW}'
print('Using artifact:', base_spec.configuration.input_data.artifact_path)
print('Using draw column:', draw_col)
print('Population size:', POP_SIZE)

Using artifact: /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf
Using draw column: draw_60
Population size: 60000


## 1) APH/PPH incidence and severity vs artifact

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [3]:
def get_art_by_age(art_key: str, sim_year: int, sim_location: str, sim_sex: str = 'Female') -> pd.DataFrame:
    art = artifact.load(art_key).reset_index()

    # Filter progressively; if a filter wipes everything out, keep previous data
    cur = art.copy()
    if 'location' in cur.columns:
        next_df = cur[cur['location'] == sim_location]
        if not next_df.empty:
            cur = next_df
    if 'sex' in cur.columns:
        next_df = cur[cur['sex'] == sim_sex]
        if not next_df.empty:
            cur = next_df
    if {'year_start', 'year_end'}.issubset(cur.columns):
        next_df = cur[(cur['year_start'] <= sim_year) & (sim_year < cur['year_end'])]
        if not next_df.empty:
            cur = next_df

    grouped = cur.groupby(['age_start', 'age_end'], as_index=False)[draw_col].mean()
    return grouped.sort_values(['age_start', 'age_end']).reset_index(drop=True)

def map_ages_to_art_values(ages: pd.Series, art_by_age: pd.DataFrame) -> pd.Series:
    intervals = pd.IntervalIndex.from_arrays(
        art_by_age['age_start'].astype(float),
        art_by_age['age_end'].astype(float),
        closed='left',
    )
    idx = intervals.get_indexer(ages.astype(float))
    out = pd.Series(np.nan, index=ages.index, dtype=float)
    matched = idx >= 0
    if matched.any():
        out.loc[matched] = art_by_age.iloc[idx[matched]][draw_col].to_numpy()
    return out

def weighted_incidence_from_artifact(
    pop: pd.DataFrame,
    art_key: str,
    eligible_mask: pd.Series,
    sim_year: int,
    sim_location: str,
    sim_sex: str = 'Female',
) -> float:
    art_by_age = get_art_by_age(art_key, sim_year=sim_year, sim_location=sim_location, sim_sex=sim_sex)
    eligible_ages = pop.loc[eligible_mask, COLUMNS.MOTHER_AGE]
    values = map_ages_to_art_values(eligible_ages, art_by_age)
    matched = values.notna()
    return float(values[matched].mean()) if matched.any() else np.nan

sim_year = int(base_spec.configuration.time.start.year)

# Single run for both APH and PPH checks (up to PPH step)
sim = make_sim('baseline')
run_to_step(sim, SIMULATION_EVENT_NAMES.POSTPARTUM_HEMORRHAGE)
pop = sim.get_population([
    COLUMNS.MOTHER_AGE,
    COLUMNS.LOCATION,
    COLUMNS.PREGNANCY_OUTCOME,
    COLUMNS.ANTEPARTUM_HEMORRHAGE,
    COLUMNS.POSTPARTUM_HEMORRHAGE,
])

sim_location = pop[COLUMNS.LOCATION].mode().iloc[0]

# Denominators must match component eligibility rules
aph_eligible = pop[COLUMNS.PREGNANCY_OUTCOME] != 'invalid'
pph_eligible = pop[COLUMNS.PREGNANCY_OUTCOME].isin(['live_birth', 'stillbirth'])

# Sim incidence
aph_cases = pop[COLUMNS.ANTEPARTUM_HEMORRHAGE].astype(bool)
pph_cases = pop[COLUMNS.POSTPARTUM_HEMORRHAGE].astype(bool)

aph_incidence_sim = float(aph_cases[aph_eligible].mean()) if int(aph_eligible.sum()) else np.nan
pph_incidence_sim = float(pph_cases[pph_eligible].mean()) if int(pph_eligible.sum()) else np.nan

# Artifact incidence weighted to simulated eligible age composition
aph_incidence_art = weighted_incidence_from_artifact(
    pop,
    MATERNAL_HEMORRHAGE.APH_INCIDENCE_RISK,
    aph_eligible,
    sim_year=sim_year,
    sim_location=sim_location,
)
pph_incidence_art = weighted_incidence_from_artifact(
    pop,
    MATERNAL_HEMORRHAGE.PPH_INCIDENCE_RISK,
    pph_eligible,
    sim_year=sim_year,
    sim_location=sim_location,
)

check1 = pd.DataFrame([
    {
        'cause': 'antepartum_hemorrhage',
        'sim_incidence': aph_incidence_sim,
        'artifact_incidence': aph_incidence_art,
        'incidence_ratio_sim_over_artifact': aph_incidence_sim / aph_incidence_art if aph_incidence_art else np.nan,
    },
    {
        'cause': 'postpartum_hemorrhage',
        'sim_incidence': pph_incidence_sim,
        'artifact_incidence': pph_incidence_art,
        'incidence_ratio_sim_over_artifact': pph_incidence_sim / pph_incidence_art if pph_incidence_art else np.nan,
    },
])

check1

2026-06-16 10:19:06.877 | INFO     | simulation_1-artifact_manager:80 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf.


2026-06-16 10:19:06.878 | INFO     | simulation_1-artifact_manager:81 - Artifact base filter terms are ['draw == 60'].


2026-06-16 10:19:06.879 | INFO     | simulation_1-artifact_manager:82 - Artifact additional filter terms are None.


/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:234: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  sub_tables = list(data.groupby(list(other_params)))
/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:234: 

/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:234: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  sub_tables = list(data.groupby(list(other_params)))
/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:234: 

...

/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:234: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  sub_tables = list(data.groupby(list(other_params)))
/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:234: 

/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:234: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  sub_tables = list(data.groupby(list(other_params)))
/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:234: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  sub_tables = list(data.groupby(list(other_params)))
/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: In a 

/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:234: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  sub_tables = list(data.groupby(list(other_params)))
/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:234: 

/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:234: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  sub_tables = list(data.groupby(list(other_params)))
/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:234: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  sub_tables = list(data.groupby(list(other_params)))
/ihme/homes/tylerdy/miniforge3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:234: FutureWarning: In a

TypeError: NonLogLinearRiskEffect.validate_rr_data.<locals>.values_are_monotonically_increasing() got an unexpected keyword argument 'include_groups'

## 2) Only severe hemorrhage cases die from hemorrhage

In [ ]:
sim_mort = make_sim('baseline')
run_to_step(sim_mort, SIMULATION_EVENT_NAMES.MORTALITY)

APH_SEVERITY_COL = f"{COLUMNS.ANTEPARTUM_HEMORRHAGE}_severity"
PPH_SEVERITY_COL = f"{COLUMNS.POSTPARTUM_HEMORRHAGE}_severity"

mort_pop = sim_mort.get_population([
    COLUMNS.MOTHER_CAUSE_OF_DEATH,
    APH_SEVERITY_COL,
    PPH_SEVERITY_COL,
])

hemo_cod = [COLUMNS.ANTEPARTUM_HEMORRHAGE, COLUMNS.POSTPARTUM_HEMORRHAGE]
dead_hemo = mort_pop[mort_pop[COLUMNS.MOTHER_CAUSE_OF_DEATH].isin(hemo_cod)]

bad_aph = dead_hemo[
    (dead_hemo[COLUMNS.MOTHER_CAUSE_OF_DEATH] == COLUMNS.ANTEPARTUM_HEMORRHAGE)
    & (dead_hemo[APH_SEVERITY_COL] != HEMORRHAGE_SEVERITY.SEVERE)
]
bad_pph = dead_hemo[
    (dead_hemo[COLUMNS.MOTHER_CAUSE_OF_DEATH] == COLUMNS.POSTPARTUM_HEMORRHAGE)
    & (dead_hemo[PPH_SEVERITY_COL] != HEMORRHAGE_SEVERITY.SEVERE)
]

assert len(bad_aph) == 0, f'{len(bad_aph)} non-severe APH deaths found'
assert len(bad_pph) == 0, f'{len(bad_pph)} non-severe PPH deaths found'
print(f'PASSED: all {len(dead_hemo)} hemorrhage deaths were severe')

## 3) Misoprostol affects postpartum hemorrhage incidence but not antepartum hemorrhage incidence, aph and pph incidence are the same, aph and pph severe incidence are the same

In [ ]:
art_pp_frac = artifact.load("cause.maternal_hemorrhage.postpartum_fraction").mean(axis=1).reset_index()
art_pp_frac_male = art_pp_frac.copy()
art_pp_frac_male["sex"] = "Male"
art_pp_frac = pd.concat([art_pp_frac, art_pp_frac_male]).set_index(["sex", "age_start", "age_end", "year_start", "year_end"]).rename(columns={0: "value"})
pop_structure = artifact.load("population.structure").xs("Ethiopia", level="location").rename(columns={"value": "pop_struc"})
weighted_pp_frac = pd.concat([art_pp_frac, pop_structure], axis=1).dropna()
weighted_pp_frac["pop_struc"] = weighted_pp_frac["pop_struc"] / weighted_pp_frac["pop_struc"].sum()
assert abs(weighted_pp_frac["pop_struc"].sum() - 1) < (.1 ** 6) # rounds to 1
weighted_pp_frac = (weighted_pp_frac["value"] * weighted_pp_frac["pop_struc"]).sum()
weighted_pp_frac


In [ ]:
def pph_summary_for_scenario(scenario: str) -> pd.DataFrame:
    sim = make_sim(scenario)
    run_to_step(sim, SIMULATION_EVENT_NAMES.POSTPARTUM_HEMORRHAGE)

    APH_SEVERITY_COL = f"{COLUMNS.ANTEPARTUM_HEMORRHAGE}_severity"
    PPH_SEVERITY_COL = f"{COLUMNS.POSTPARTUM_HEMORRHAGE}_severity"



    pop = sim.get_population([
        COLUMNS.POSTPARTUM_HEMORRHAGE,
        COLUMNS.ANTEPARTUM_HEMORRHAGE,
        COLUMNS.MISOPROSTOL_AVAILABLE,
        COLUMNS.DELIVERY_FACILITY_TYPE,
        APH_SEVERITY_COL,
        PPH_SEVERITY_COL,
    ])

    out = []
    aph = float(pop[COLUMNS.ANTEPARTUM_HEMORRHAGE].mean())
    pph = float(pop[COLUMNS.POSTPARTUM_HEMORRHAGE].mean())
    out.append({
        'scenario': scenario, 
        'group': 'overall', 
        'aph_incidence': aph, 
        'pph_incidence': pph, 
        'aph_severity': float((pop[APH_SEVERITY_COL].replace({'none': np.nan, 'moderate':'False', 'severe': 'True'}).dropna() == 'True').mean()), 
        'pph_severity': float((pop[PPH_SEVERITY_COL].replace({'none': np.nan, 'moderate':'False', 'severe': 'True'}).dropna() == 'True').mean()),
        'n': len(pop)
        })

    for value in [True, False]:
        sub = pop[pop[COLUMNS.MISOPROSTOL_AVAILABLE] == value]
        if len(sub) == 0:
            continue
        out.append({
            'scenario': scenario,
            'group': f'misoprostol_available={value}',
            'aph_incidence': float(sub[COLUMNS.ANTEPARTUM_HEMORRHAGE].mean()),
            'pph_incidence': float(sub[COLUMNS.POSTPARTUM_HEMORRHAGE].mean()),
            'aph_severity': float((sub[APH_SEVERITY_COL].replace({'none': np.nan, 'moderate':'False', 'severe': 'True'}).dropna() == 'True').mean()),
            'pph_severity': float((sub[PPH_SEVERITY_COL].replace({'none': np.nan, 'moderate':'False', 'severe': 'True'}).dropna() == 'True').mean()),
            'n': len(sub),
        })

    home = pop[pop[COLUMNS.DELIVERY_FACILITY_TYPE] == 'home']
    if len(home):
        out.append({
            'scenario': scenario,
            'group': 'home_only',
            'aph_incidence': float(home[COLUMNS.ANTEPARTUM_HEMORRHAGE].mean()),
            'pph_incidence': float(home[COLUMNS.POSTPARTUM_HEMORRHAGE].mean()),
            'aph_severity': float((home[APH_SEVERITY_COL].replace({'none': np.nan, 'moderate':'False', 'severe': 'True'}).dropna() == 'True').mean()),
            'pph_severity': float((home[PPH_SEVERITY_COL].replace({'none': np.nan, 'moderate':'False', 'severe': 'True'}).dropna() == 'True').mean()),
            'n': len(home),
        })

    return pd.DataFrame(out)

baseline_pph = pph_summary_for_scenario('baseline')
miso_vv_pph = pph_summary_for_scenario('misoprostol_vv')

check4 = pd.concat([baseline_pph, miso_vv_pph], ignore_index=True)
check4

In [ ]:
# Directional check: compare home-birth PPH incidence across scenarios.
# Misoprostol is only available to home births so that is the right comparison group.
baseline_home = float(check4.query("scenario == 'baseline' and group == 'home_only'")['pph_incidence'].iloc[0])
miso_home = float(check4.query("scenario == 'misoprostol_vv' and group == 'home_only'")['pph_incidence'].iloc[0])

pd.DataFrame([
    {'metric': 'baseline_home_birth_pph_incidence', 'value': baseline_home},
    {'metric': 'misoprostol_vv_home_birth_pph_incidence', 'value': miso_home},
    {'metric': 'directional_pass', 'value': miso_home < baseline_home},
])